In [18]:
import pandas as pd

df = pd.read_csv('/content/final1.csv')
print(df.head().to_csv(index=False))

title,body,is_fraud,fraud_type,transaction_crypto_scam,transaction_upi_fraud,transaction_card_fraud,transaction_bank_transfer,commerce_nondelivery,commerce_fake_seller,credential_phishing,social_authority_scam,social_urgency_scam,meta_victim_story,meta_fraud_question,payment_method,fraud_channel,currency,urgency_level,urgency,fear,authority,reward,impersonated_entity_clean_crypto_platform,impersonated_entity_clean_delivery_courier,impersonated_entity_clean_ecommerce,impersonated_entity_clean_education,impersonated_entity_clean_employment_agency,impersonated_entity_clean_financial_institution,impersonated_entity_clean_friend_family,impersonated_entity_clean_government,impersonated_entity_clean_healthcare,impersonated_entity_clean_law_enforcement,impersonated_entity_clean_non_profit,impersonated_entity_clean_public_figure,impersonated_entity_clean_service_provider,impersonated_entity_clean_tech_company,impersonated_entity_clean_telecom,impersonated_entity_clean_unknown,impersonated_entit

In [19]:
is_fraud_counts = df['is_fraud'].value_counts()
print("Occurrences of each value in 'is_fraud' column:")
print(is_fraud_counts)

Occurrences of each value in 'is_fraud' column:
is_fraud
 1    2551
-1    1412
 0     107
Name: count, dtype: int64


In [15]:
print(f"Number of rows before deletion: {len(df)}")
df = df[df['is_fraud'] != -1].copy()
print(f"Number of rows after deleting '-1' from 'is_fraud': {len(df)}")

is_fraud_counts = df['is_fraud'].value_counts()
print("Occurrences of each value in 'is_fraud' column:")
print(is_fraud_counts)

Number of rows before deletion: 2658
Number of rows after deleting '-1' from 'is_fraud': 2658
Occurrences of each value in 'is_fraud' column:
is_fraud
1    2551
0     107
Name: count, dtype: int64


#Run 1 : Batch size 50, epochs 300

In [5]:
import pandas as pd
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# ── 1. Load your data ────────────────────────────────────────────────────────
# df = pd.read_csv("your_data.csv")  # <-- update this path

# ── 2. Pre-process: replace "unknown" strings with NaN ──────────────────────
df.replace("unknown", pd.NA, inplace=True)

# ── 3. Drop high-cardinality text columns (not suitable for CTGAN) ───────────
TEXT_COLS = ["title", "body"]
not_fraud_df = df[df["is_fraud"] == 0].drop(columns=TEXT_COLS).copy()

# ── 4. Build metadata ────────────────────────────────────────────────────────
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(not_fraud_df)

# Fix: amount_numeric may be detected as numerical but has mixed types
# If you get errors, uncomment the line below:
# metadata.update_column("amount_numeric", sdtype="categorical")

# ── 5. Train CTGAN on not-fraud rows only ───────────────────────────────────
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=300,
    batch_size=50,      # small batch since not-fraud only has 107 rows
    verbose=True
)
synthesizer.fit(not_fraud_df)

# ── 6. Generate ~500 synthetic not-fraud samples ─────────────────────────────
synthetic_not_fraud = synthesizer.sample(num_rows=500)
synthetic_not_fraud["is_fraud"] = 0  # ensure label is correct

# Add back empty text columns so schema matches original df
for col in TEXT_COLS:
    synthetic_not_fraud[col] = pd.NA

# ── 7. Combine with original data ────────────────────────────────────────────
balanced_df = pd.concat([df, synthetic_not_fraud], ignore_index=True)

print(f"Original distribution:\n{df['is_fraud'].value_counts()}")
print(f"\nNew distribution:\n{balanced_df['is_fraud'].value_counts()}")

# ── 8. Save ──────────────────────────────────────────────────────────────────
balanced_df.to_csv("balanced_data.csv", index=False)
synthesizer.save("ctgan_not_fraud_model.pkl")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+02.90) | Discrim. (-00.31): 100%|██████████| 300/300 [00:33<00:00,  8.89it/s]


Original distribution:
is_fraud
1    2551
0     107
Name: count, dtype: int64

New distribution:
is_fraud
1    2551
0     607
Name: count, dtype: int64


# Validation for Run 1

In [6]:
import pandas as pd
import numpy as np

real = not_fraud_df  # your original 107 not-fraud rows
synthetic = synthetic_not_fraud.drop(columns=["title", "body"])

# Compare means and distributions for numeric columns
print("=== Numeric Column Comparison ===")
numeric_cols = real.select_dtypes(include=[np.number]).columns.tolist()
comparison = pd.DataFrame({
    "real_mean": real[numeric_cols].mean(),
    "synthetic_mean": synthetic[numeric_cols].mean(),
    "real_std": real[numeric_cols].std(),
    "synthetic_std": synthetic[numeric_cols].std()
})
print(comparison)

# Compare value counts for categorical columns
print("\n=== Categorical Column Comparison ===")
cat_cols = real.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    print(f"\n-- {col} --")
    real_pct = real[col].value_counts(normalize=True).rename("real_%")
    syn_pct = synthetic[col].value_counts(normalize=True).rename("synthetic_%")
    print(pd.concat([real_pct, syn_pct], axis=1).fillna(0).round(3))

=== Numeric Column Comparison ===
                           real_mean  synthetic_mean  real_std  synthetic_std
is_fraud                    0.000000          0.0000  0.000000       0.000000
transaction_crypto_scam     0.046729          0.0520  0.212051       0.222249
transaction_upi_fraud       0.000000          0.0000  0.000000       0.000000
transaction_card_fraud      0.028037          0.0220  0.165856       0.146830
transaction_bank_transfer   0.009346          0.0100  0.096674       0.099598
commerce_nondelivery        0.168224          0.2300  0.375826       0.421254
commerce_fake_seller        0.028037          0.0540  0.165856       0.226244
credential_phishing         0.018692          0.0180  0.136071       0.133084
social_authority_scam       0.065421          0.0520  0.248430       0.222249
social_urgency_scam         0.018692          0.0280  0.136071       0.165138
meta_victim_story           0.168224          0.1220  0.375826       0.327614
meta_fraud_question         0.

#Run 2: Batch size 20, epochs 1000

In [16]:
import pandas as pd
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

# ── 1. Load your data ────────────────────────────────────────────────────────
# df = pd.read_csv("your_data.csv")  # <-- update this path

# ── 2. Pre-process: replace "unknown" strings with NaN ──────────────────────
df.replace("unknown", pd.NA, inplace=True)

# ── 3. Drop high-cardinality text columns (not suitable for CTGAN) ───────────
TEXT_COLS = ["title", "body"]
not_fraud_df = df[df["is_fraud"] == 0].drop(columns=TEXT_COLS).copy()

# ── 4. Build metadata ────────────────────────────────────────────────────────
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(not_fraud_df)

# Fix: amount_numeric may be detected as numerical but has mixed types
# If you get errors, uncomment the line below:
# metadata.update_column("amount_numeric", sdtype="categorical")

# ── 5. Train CTGAN on not-fraud rows only ───────────────────────────────────
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=1000,
    batch_size=20,      # small batch since not-fraud only has 107 rows
    verbose=True
)
synthesizer.fit(not_fraud_df)

# ── 6. Generate ~500 synthetic not-fraud samples ─────────────────────────────
synthetic_not_fraud = synthesizer.sample(num_rows=500)
synthetic_not_fraud["is_fraud"] = 0  # ensure label is correct

# Add back empty text columns so schema matches original df
for col in TEXT_COLS:
    synthetic_not_fraud[col] = pd.NA

# ── 7. Combine with original data ────────────────────────────────────────────
balanced_df = pd.concat([df, synthetic_not_fraud], ignore_index=True)

print(f"Original distribution:\n{df['is_fraud'].value_counts()}")
print(f"\nNew distribution:\n{balanced_df['is_fraud'].value_counts()}")

# ── 8. Save ──────────────────────────────────────────────────────────────────
balanced_df.to_csv("balanced_data.csv", index=False)
synthesizer.save("ctgan_not_fraud_model.pkl")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (+01.25) | Discrim. (+00.69): 100%|██████████| 1000/1000 [04:06<00:00,  4.05it/s]


Original distribution:
is_fraud
1    2551
0     107
Name: count, dtype: int64

New distribution:
is_fraud
1    2551
0     607
Name: count, dtype: int64


# Validation for Run 2

In [17]:
import pandas as pd
import numpy as np

real = not_fraud_df  # your original 107 not-fraud rows
synthetic = synthetic_not_fraud.drop(columns=["title", "body"])

# Compare means and distributions for numeric columns
print("=== Numeric Column Comparison ===")
numeric_cols = real.select_dtypes(include=[np.number]).columns.tolist()
comparison = pd.DataFrame({
    "real_mean": real[numeric_cols].mean(),
    "synthetic_mean": synthetic[numeric_cols].mean(),
    "real_std": real[numeric_cols].std(),
    "synthetic_std": synthetic[numeric_cols].std()
})
print(comparison)

# Compare value counts for categorical columns
print("\n=== Categorical Column Comparison ===")
cat_cols = real.select_dtypes(include=["object", "category"]).columns.tolist()
for col in cat_cols:
    print(f"\n-- {col} --")
    real_pct = real[col].value_counts(normalize=True).rename("real_%")
    syn_pct = synthetic[col].value_counts(normalize=True).rename("synthetic_%")
    print(pd.concat([real_pct, syn_pct], axis=1).fillna(0).round(3))

=== Numeric Column Comparison ===
                           real_mean  synthetic_mean  real_std  synthetic_std
is_fraud                    0.000000          0.0000  0.000000       0.000000
transaction_crypto_scam     0.046729          0.0360  0.212051       0.186477
transaction_upi_fraud       0.000000          0.0000  0.000000       0.000000
transaction_card_fraud      0.028037          0.0500  0.165856       0.218163
transaction_bank_transfer   0.009346          0.0100  0.096674       0.099598
commerce_nondelivery        0.168224          0.1460  0.375826       0.353460
commerce_fake_seller        0.028037          0.0260  0.165856       0.159295
credential_phishing         0.018692          0.0320  0.136071       0.176176
social_authority_scam       0.065421          0.0520  0.248430       0.222249
social_urgency_scam         0.018692          0.0200  0.136071       0.140140
meta_victim_story           0.168224          0.3140  0.375826       0.464581
meta_fraud_question         0.